In [ ]:
%run ./_common

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
np.set_printoptions(precision=2)

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

In [ ]:
SA = SWSAcademyHelper()
SA.init()

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import split, col, monotonically_increasing_id
from pyspark.ml import Pipeline
from pyspark.ml.feature import Word2Vec, VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.functions import vector_to_array

import mlflow
from databricks.feature_engineering import FeatureEngineeringClient

In [ ]:
airbnb_base_path = SA.datasets.airbnb.base_path
df = spark.read.format("csv").option("header", "true").load(f"{airbnb_base_path}/sf-listings/airbnb-cleaned-mlflow.csv")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SA.catalog_name}.{SA.schema_name}.airbnb_cleaned_mlflow")

In [ ]:
source_table = f"{SA.catalog_name}.{SA.schema_name}.airbnb_cleaned_mlflow"
feature_table_name = f"{SA.catalog_name}.{SA.schema_name}.airbnb_features"
df = spark.table(source_table)

row_count = df.count()
mlflow.log_metric("airbnb_row_count", row_count)
assert row_count > 0, f"Source table {source_table} is empty."

display(df.limit(5))

In [ ]:
all_numeric_cols = [
    "host_total_listings_count",
    "neighbourhood_cleansed",
    "zipcode",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "bed_type",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    # price is target (used to derive label), not a feature
]

required_cols = all_numeric_cols + ["price"]

missing = [c for c in required_cols if c not in df.columns]
assert not missing, f"Missing required columns: {missing}"

# Ensure numeric types (guard against accidental schema drift)
for c in all_numeric_cols + ["price"]:
    df = df.withColumn(c, col(c).cast("double"))

# Drop rows with null price
df = df.dropna(subset=["price"])

In [ ]:
p75 = df.approxQuantile("price", [0.75], 0.01)[0]
mlflow.log_metric("price_p75", float(p75))

labeled_df = df.withColumn("is_high_price", (col("price") >= F.lit(p75)).cast("int"))

In [ ]:
assembler = VectorAssembler(
    inputCols=all_numeric_cols,
    outputCol="features_vec"
)

features_df = assembler.transform(labeled_df)

# Add a stable (surrogate) primary key and a convenient array form of the vector
features_df = (
    features_df
    .withColumn("id", monotonically_increasing_id())
    .withColumn("features_array", vector_to_array("features_vec"))
    .select(
        "id",
        *all_numeric_cols,
        "price",
        "is_high_price",
        "features_vec",
        "features_array"
    )
)

mlflow.log_metric("features_row_count", features_df.count())

# Quick sanity check: vector length should be consistent
vec_stats = features_df.selectExpr("size(features_array) as len").groupBy("len").count()
display(vec_stats)

In [ ]:
# Drop VectorUDT column; keep only SQL-friendly types
clean_features_df = (
    features_df
    .drop("features_vec")
    # (Optional) reorder for readability
    .select(
        "id",
        "price",
        "is_high_price",
        "features_array",
        *[c for c in features_df.columns
          if c not in {"id", "price", "is_high_price", "features_vec", "features_array"}]
    )
)

fe = FeatureEngineeringClient()

try:
    fe.drop_table(name=feature_table_name)
except Exception as e:
    print("Skip drop if not exists:", e)

fe.create_table(
    name=feature_table_name,
    primary_keys=["id"],
    df=clean_features_df,
    description="Airbnb numeric features with high-price label (P75 of price); features stored as array<double>"
)

display(spark.table(feature_table_name).limit(5))

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SA.catalog_name}.{SA.schema_name}.ray_logs")
print(f"Ensured volume exists for Ray logs at {SA.catalog_name}.{SA.schema_name}.ray_logs")

In [ ]:
# Load Airbnb dataset from Unity Catalog
df = spark.read.table(f"{SA.catalog_name}.{SA.schema_name}.airbnb_cleaned_mlflow")

# Display basic summary statistics
display(df.describe())

In [ ]:
from pyspark.sql.types import (
    DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType, StringType
)
from pyspark.sql import functions as F

# Start from original df
df_raw = df

# Columns that are numeric in *content* but stored as string
numeric_candidate_cols = [
    "host_total_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "price",  # label
]

# Only keep the ones that actually exist in this table
numeric_candidate_cols = [c for c in numeric_candidate_cols if c in df_raw.columns]

print("Numeric candidate columns (string → double):", numeric_candidate_cols)

# Cast them to double
df = df_raw
for c in numeric_candidate_cols:
    df = df.withColumn(c, F.col(c).cast("double"))

# Now re-identify numeric columns from the *cast* DataFrame
numeric_types = (DoubleType, FloatType, IntegerType, LongType, ShortType, DecimalType)
label_col = "price"

numeric_cols_all = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, numeric_types)
]

print("All numeric columns after cast:", numeric_cols_all)

# Features = all numeric except label
feature_cols = [c for c in numeric_cols_all if c != label_col]

print("Feature columns (used for model):", feature_cols)

# Guard
if not feature_cols:
    raise ValueError(
        "feature_cols came out empty even after casting. "
        "Check the schema or candidate column list."
    )

In [ ]:
# Clean label and create log(label) for regression target
df_clean = (
    df
    .filter(F.col(label_col).isNotNull() & (F.col(label_col) > 0))
    .withColumn("label", F.log1p(F.col(label_col)))
)

train_sdf, val_sdf, test_sdf = df_clean.randomSplit([0.7, 0.15, 0.15], seed=42)

print("Train:", train_sdf.count(), "Val:", val_sdf.count(), "Test:", test_sdf.count())

In [ ]:
# Spark ML models logged or loaded with MLflow need a temporary folder. The default (/dbfs/tmp) is blocked when the
# workspace disables the DBFS root, so use a Unity Catalog volume in the learner schema instead.
import os
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SA.catalog_name}.{SA.schema_name}.mlflow_tmp")
os.environ["MLFLOW_DFS_TMP"] = f"/Volumes/{SA.catalog_name}/{SA.schema_name}/mlflow_tmp"

# On dedicated (single-user) clusters MLflow loads Spark ML models through the DBFS root (/dbfs/...), which is blocked
# when the workspace disables it. Use MLflow's own Unity Catalog volume loader instead.
import mlflow.spark as _mlflow_spark
if hasattr(_mlflow_spark, "_load_model_databricks_uc_volume"):
    _mlflow_spark._load_model_databricks_dbfs = _mlflow_spark._load_model_databricks_uc_volume

In [ ]:
# The Airbnb file has a few malformed rows (amenities text spilling into numeric columns). Databricks Runtime 17+
# turns on ANSI SQL mode by default, which stops on such values; the course was built for 16.4, where they become
# null. Use the 16.4 behaviour on any runtime.
try:
    spark.conf.set("spark.sql.ansi.enabled", "false")
except Exception:
    pass